# raw → `df.parquet`

`data-v2/raw/`만 읽어서 모델 입력 `data-v2/input/df.parquet`를 만듭니다. 로직은 모두 이 노트북 안에 있고, 다른 모듈이나 중간 캐시(processed/)는 쓰지 않습니다. 실행할 때마다 SUVI NetCDF에서 다시 추출합니다(12 worker로 약 30분).

| 단계 | 내용 |
|---|---|
| 0 | 설정과 시간 칸 규칙 |
| 1 | ACE 월별 txt → 시간 칸 표, 정답 `y` |
| 2 | ACE 피처 |
| 3 | SUVI NetCDF → 프레임별 셀 통계·면적 |
| 4 | SUVI 피처(프레임, `img_age_h`, strip) |
| 5 | ENLIL 피처 |
| 6 | df 조립·저장 |
| 7 | 검증 |

**규칙**
- 입수지연·결측·이상치 처리는 하지 않습니다. 앞값 채움, 이전 프레임으로 되돌아가기, stale/cover 마스크, 값 자르기(clip)·로그 변환이 없고, 피처는 원문 값에서 계산만 합니다.
- **시간 칸**: 시각 t는 칸 `floor(t, 1h) + 1h`에 들어갑니다. 즉 [H−1h, H) → H이고, 19:49·19:59·19:00은 모두 20시 칸입니다. 한 칸에 여러 개면 가장 늦은 것을 씁니다. 발표 시각이 따로 없는 소스도 데이터의 시각에 같은 규칙을 씁니다. origin O의 피처는 칸 ≤ O만 씁니다.
- **정답 `y`**: 공식 채점(`metric/metric.ipynb`)과 같습니다. target_time T의 `y`는 T 라벨 SWEPAM 속도이고, 시각을 이동하지 않습니다. `status == 0`이고 유한한 양수일 때만 값이 있고, 나머지는 NaN이며 채우지 않습니다.
- **ACE 피처 값**: `status ≠ 0`이거나 결측 표시값(제품 헤더의 `Missing data values`)이면 NaN입니다.
- 이름에 `ch`를 쓰지 않습니다. SUVI에서 나온 열은 `suvi_`로 시작합니다(`strip_*`, `img_age_h`는 예전 이름 그대로).

## 0. 설정

In [1]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import json, re, time, warnings, multiprocessing
from concurrent.futures import ProcessPoolExecutor
from datetime import datetime, timezone
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from scipy import ndimage

warnings.filterwarnings("ignore", category=RuntimeWarning)   # 전부 NaN인 창의 평균 등
pd.set_option("display.width", 200, "display.max_columns", 40)

ROOT = Path("/home/t-lab01/sunrun")
DATA = ROOT / "data-v2"
ACE_DIR = DATA / "raw/ACE/monthly-txt"
NC_DIR = DATA / "raw/SUVI/nc/g18"
ENLIL_DIR = DATA / "raw/ENLIL/donki"
OUT = Path(os.environ.get("DF_OUT", DATA / "input"))       # df.parquet, MANIFEST.json
SMOKE = os.environ.get("DF_SMOKE") == "1"                   # 짧은 구간만 (확인용)
WORKERS = int(os.environ.get("DF_WORKERS", "12"))

H = 72
MASK_HOURS = 144
HOUR = pd.Timedelta(hours=1)
PERIODS = {  # 이름 -> (첫 query origin, 마지막 query origin, 학습 cutoff)
    "F1": ("2025-12-31T23:00Z", "2026-03-28T23:00Z", "2025-12-29T00:00Z"),
    "F2": ("2026-03-31T23:00Z", "2026-06-27T23:00Z", "2026-03-29T00:00Z"),
    "F3": ("2026-06-30T23:00Z", "2026-09-26T23:00Z", "2026-06-29T00:00Z"),
}
QUERY_SPAN = ("2025-12-31T23:00Z", "2026-09-26T23:00Z")
TRAIN_SPAN = ("2022-10-01T00:00Z", str(pd.Timestamp(PERIODS["F3"][2]) - pd.Timedelta(hours=MASK_HOURS)))
if SMOKE:
    TRAIN_SPAN = ("2026-05-20T00:00Z", "2026-05-21T23:00Z")
    QUERY_SPAN = ("2026-05-28T00:00Z", "2026-05-29T23:00Z")

train_origins = pd.date_range(*TRAIN_SPAN, freq="h")
query_origins = pd.date_range(*QUERY_SPAN, freq="h")
all_origins = train_origins.union(query_origins)


def bin_hour(t):
    # 시각 -> 시간 칸: [H-1h, H) -> H
    return pd.DatetimeIndex(pd.to_datetime(t, utc=True)).floor("h") + HOUR


for raw_t, want in [("2026-05-01T19:49Z", "2026-05-01T20:00Z"), ("2026-05-01T19:59:59Z", "2026-05-01T20:00Z"),
                    ("2026-05-01T19:00Z", "2026-05-01T20:00Z"), ("2026-05-01T20:00Z", "2026-05-01T21:00Z")]:
    assert bin_hour([raw_t])[0] == pd.Timestamp(want), (raw_t, want)

print("SMOKE" if SMOKE else "FULL", "| train origin", len(train_origins), TRAIN_SPAN, "| query origin", len(query_origins), QUERY_SPAN)
print("출력:", OUT, "| workers:", WORKERS)

FULL | train origin 32665 ('2022-10-01T00:00Z', '2026-06-23 00:00:00+00:00') | query origin 6457 ('2025-12-31T23:00Z', '2026-09-26T23:00Z')
출력: /home/t-lab01/sunrun/data-v2/input | workers: 12


## 1. ACE: 월별 txt → 시간 칸 표, 정답 `y`

NASA 월별 1h 파일 5종(`swepam, mag, epam, sis, loc`)을 읽습니다. 각 행의 라벨 시각은 `YR MO DA HHMM`(UT)입니다.
- 피처용 표: 라벨 시각 t → 칸 `bin_hour(t)` = t + 1h. 값은 그 값의 status가 0이 아니거나 결측 표시값이면 NaN입니다(loc에는 status 열이 없음).
- `y`용 표: 라벨 시각 그대로, SWEPAM `status == 0`이고 유한한 양수인 속도.
- 같은 라벨 시각이 두 번 나오면(월 경계 등) 나중 파일의 행을 쓰고, 값이 서로 다른 중복은 개수를 기록합니다.

In [2]:
# 제품 -> (시각 뒤 필드 순서, 결측 표시값). 필드: (열 이름 | None, status 열 이름 | None)
# 시각 필드 6개: YR MO DA HHMM MJD SEC
ACE_SPEC = {
    "swepam": ([("S", None), ("ace_density_cm3", "S"), ("ace_speed_kms", "S"), ("ace_temperature_k", "S")],
               {"ace_density_cm3": -9999.9, "ace_speed_kms": -9999.9, "ace_temperature_k": -1.0e5}),
    "mag": ([("S", None), ("ace_bx_gsm_nt", "S"), ("ace_by_gsm_nt", "S"), ("ace_bz_gsm_nt", "S"), ("ace_bt_nt", "S"),
             ("ace_lat_gsm_deg", "S"), ("ace_lon_gsm_deg", "S")], -999.9),
    "epam": ([("Se", None), ("ace_epam_e_channel1", "Se"), ("ace_epam_e_channel2", "Se"), ("Sp", None),
              ("ace_epam_p_channel1", "Sp"), ("ace_epam_p_channel2", "Sp"), ("ace_epam_p_channel3", "Sp"),
              ("ace_epam_p_channel4", "Sp"), ("ace_epam_p_channel5", "Sp"), ("ace_epam_anis_index", "Sp")],
             {"ace_epam_anis_index": -1.0, "*": -1.0e5}),
    "sis": ([("S10", None), ("ace_sis_gt10_flux", "S10"), ("S30", None), ("ace_sis_gt30_flux", "S30")], -1.0e5),
    "loc": ([("ace_x_gse_re", None), ("ace_y_gse_re", None), ("ace_z_gse_re", None)], -999.9),
}
ACE_LABEL = {  # 원문 헤더의 이름 (문서용)
    "ace_density_cm3": "SWEPAM proton density", "ace_speed_kms": "SWEPAM bulk speed", "ace_temperature_k": "SWEPAM ion temperature",
    "ace_bx_gsm_nt": "MAG Bx GSM", "ace_by_gsm_nt": "MAG By GSM", "ace_bz_gsm_nt": "MAG Bz GSM", "ace_bt_nt": "MAG Bt",
    "ace_lat_gsm_deg": "MAG lat GSM", "ace_lon_gsm_deg": "MAG long GSM",
    "ace_epam_e_channel1": "EPAM electron 38-53 keV", "ace_epam_e_channel2": "EPAM electron 175-315 keV",
    "ace_epam_p_channel1": "EPAM proton 47-68 keV", "ace_epam_p_channel2": "EPAM proton 115-195 keV",
    "ace_epam_p_channel3": "EPAM proton 310-580 keV", "ace_epam_p_channel4": "EPAM proton 795-1193 keV",
    "ace_epam_p_channel5": "EPAM proton 1060-1900 keV", "ace_epam_anis_index": "EPAM anisotropy index",
    "ace_sis_gt10_flux": "SIS proton > 10 MeV", "ace_sis_gt30_flux": "SIS proton > 30 MeV",
    "ace_x_gse_re": "LOC X GSE", "ace_y_gse_re": "LOC Y GSE", "ace_z_gse_re": "LOC Z GSE",
}


def read_ace(product):
    fields, missing = ACE_SPEC[product]
    names = [n for n, _ in fields]
    rows, files = [], sorted(ACE_DIR.glob(f"*_ace_{product}_1h.txt"))
    for order, path in enumerate(files):
        for line in path.read_text().splitlines():
            if not line or line[0] in ":#":
                continue
            parts = line.split()
            if len(parts) != 6 + len(fields):
                raise ValueError(f"{path.name}: 필드 수 {len(parts)}: {line!r}")
            rows.append([order] + parts[:4] + parts[6:])
    raw = pd.DataFrame(rows, columns=["file", "yr", "mo", "da", "hhmm"] + names)
    hhmm = raw["hhmm"].astype(int)
    t = pd.to_datetime(dict(year=raw.yr.astype(int), month=raw.mo.astype(int), day=raw.da.astype(int)), utc=True)
    raw.insert(0, "label_utc", t + pd.to_timedelta(hhmm // 100, unit="h") + pd.to_timedelta(hhmm % 100, unit="m"))
    vals = raw[names].astype(float)
    vals.insert(0, "label_utc", raw["label_utc"])
    vals.insert(1, "file", raw["file"])
    conflict = int(vals.drop(columns="file").drop_duplicates().duplicated("label_utc").sum())
    vals = vals.sort_values(["label_utc", "file"]).drop_duplicates("label_utc", keep="last").set_index("label_utc")
    clean = pd.DataFrame(index=vals.index)
    for name, status in fields:
        if not name.startswith("ace_"):
            continue
        v = vals[name].copy()
        miss = missing.get(name, missing.get("*")) if isinstance(missing, dict) else missing
        bad = np.isclose(v, miss, rtol=0, atol=1e-6) | ~np.isfinite(v)
        if status is not None:
            bad |= vals[status].ne(0)
        clean[name] = v.where(~bad)
    return clean, vals, {"files": len(files), "rows": len(rows), "labels": len(vals), "conflicting_duplicates": conflict}


ace_parts, ace_raw, ace_info = {}, {}, {}
for product in ACE_SPEC:
    ace_parts[product], ace_raw[product], ace_info[product] = read_ace(product)
ace_label = pd.concat(ace_parts.values(), axis=1).sort_index()      # 라벨 시각 기준, status/결측 표시 -> NaN
assert ace_label.index.is_unique

# 피처용: 라벨 시각 -> 칸 (라벨이 모두 정각이라 칸 = 라벨 + 1h, 한 칸에 한 행)
ace_bin = ace_label.copy()
ace_bin.index = bin_hour(ace_bin.index)
ace_bin = ace_bin[~ace_bin.index.duplicated(keep="last")]
ace_bin = ace_bin.reindex(pd.date_range(ace_bin.index.min(), ace_bin.index.max(), freq="h"))

# y용: 라벨 시각 그대로, SWEPAM status == 0 이고 유한한 양수
sw = ace_raw["swepam"]
y_series = sw["ace_speed_kms"].where(sw["S"].eq(0) & np.isfinite(sw["ace_speed_kms"]) & (sw["ace_speed_kms"] > 0))
y_series = y_series.reindex(pd.date_range(y_series.index.min(), y_series.index.max(), freq="h"))

display(pd.DataFrame(ace_info).T)
print("ACE 라벨 시각:", ace_label.index.min(), "~", ace_label.index.max(), "| 칸:", ace_bin.index.min(), "~", ace_bin.index.max())
print("y 유효:", int(y_series.notna().sum()), "/", len(y_series), f"({y_series.notna().mean():.1%})")
display(pd.DataFrame({"원문": ACE_LABEL, "NaN 비율": ace_label.isna().mean().round(3)}))

,files,rows,labels,conflicting_duplicates
swepam,52,37305,37305,0
mag,52,37305,37305,0
epam,52,37305,37305,0
sis,52,37305,37305,0
loc,52,37320,37320,0


ACE 라벨 시각: 2022-08-01 00:00:00+00:00 ~ 2026-11-02 23:00:00+00:00 | 칸: 2022-08-01 01:00:00+00:00 ~ 2026-11-03 00:00:00+00:00
y 유효: 25169 / 37305 (67.5%)


,원문,NaN 비율
ace_density_cm3,SWEPAM proton density,0.323
ace_speed_kms,SWEPAM bulk speed,0.326
ace_temperature_k,SWEPAM ion temperature,0.326
ace_bx_gsm_nt,MAG Bx GSM,0.174
ace_by_gsm_nt,MAG By GSM,0.174
ace_bz_gsm_nt,MAG Bz GSM,0.174
ace_bt_nt,MAG Bt,0.174
ace_lat_gsm_deg,MAG lat GSM,0.174
ace_lon_gsm_deg,MAG long GSM,0.174
ace_epam_e_channel1,EPAM electron 38-53 keV,0.144


## 2. ACE 피처

origin O에서 칸 ≤ O만 씁니다(칸 O = 라벨 O−1h 행).
- 22개 변수마다 `<변수>_last`(칸 O), `_m6`(칸 O−5…O 평균), `_m24`(칸 O−23…O 평균). 평균은 창 안의 NaN을 빼고 계산하고, 창이 전부 NaN이면 NaN입니다.
- 속도 파생: `v_lag{0,1,3,6,12,24,48,72,120}` = 칸 O−k의 속도, `v_m{6,24,72}`(평균), `v_sd{6,24,72}`(표준편차, 값 2개 이상).
- `speed_d24 = ace_speed_kms_m6(O) − ace_speed_kms_m6(O−24h)`, `epam_p1_d6 = ace_epam_p_channel1_m6(O) − ace_epam_p_channel1_m6(O−6h)`.
- 예전 정의와 다른 점: `_last`의 앞값 채움(최대 6h)을 하지 않고, EPAM/SIS의 log10과 1e-3 clip을 하지 않습니다. 속도는 채워진 정답 시계열이 아니라 위 ACE 칸 값(status 0)입니다.

In [3]:
ACE_VARS = list(ace_label.columns)
f = {}
for c in ACE_VARS:
    x = ace_bin[c]
    f[f"{c}_last"] = x
    f[f"{c}_m6"] = x.rolling(6, min_periods=1).mean()
    f[f"{c}_m24"] = x.rolling(24, min_periods=1).mean()
speed = ace_bin["ace_speed_kms"]
v = {f"v_lag{k}": speed.shift(k) for k in (0, 1, 3, 6, 12, 24, 48, 72, 120)}
for w in (6, 24, 72):
    v[f"v_m{w}"] = speed.rolling(w, min_periods=1).mean()
    v[f"v_sd{w}"] = speed.rolling(w, min_periods=2).std()
f["speed_d24"] = f["ace_speed_kms_m6"] - f["ace_speed_kms_m6"].shift(24)
f["epam_p1_d6"] = f["ace_epam_p_channel1_m6"] - f["ace_epam_p_channel1_m6"].shift(6)
ace_feat = pd.concat([pd.DataFrame(v), pd.DataFrame(f)], axis=1).reindex(all_origins)
print("ACE 피처:", ace_feat.shape[1], "열 | origin", len(ace_feat))
ace_feat.iloc[:3, :12]

ACE 피처: 83 열 | origin 34968


,v_lag0,v_lag1,v_lag3,v_lag6,v_lag12,v_lag24,v_lag48,v_lag72,v_lag120,v_m6,v_sd6,v_m24
2022-10-01 00:00:00+00:00,NaN,NaN,414.0,451.8,501.7,484.6,462.4,605.0,300.9,426.000000,16.970563,490.83125
2022-10-01 01:00:00+00:00,445.5,NaN,NaN,NaN,NaN,478.9,466.0,611.6,NaN,432.500000,16.454483,488.74375
2022-10-01 02:00:00+00:00,430.9,445.5,NaN,438.0,459.1,476.9,461.2,NaN,293.9,430.133333,15.763988,485.86875


## 3. SUVI: NetCDF → 프레임별 셀 통계·면적

`raw/SUVI/nc/g18/`의 GOES-18 Fe195 L1b 파일을 씁니다. `fetch.py`가 시간마다 한 장씩 저장했고, 파일 이름의 `_e…`(관측 끝 `obs_end`)로 칸 `bin_hour(obs_end)`를 정합니다. 한 칸에 파일이 여러 개면 `obs_end`가 가장 늦은 것을 씁니다.

프레임마다 두 가지를 한 번에 계산합니다(유효 픽셀 = 디스크 0.98 R☉ 안, `DQF == 0`, 유한한 비음수 radiance).
- **셀 통계 148열** `suvi_disk_*`, `suvi_lonII_latJ_*`: 디스크 전체와 경도 12칸 × 위도 3칸마다 log1p radiance의 중앙값(`log_median`)·10% 분위(`log_q10`), 어두운 픽셀 비율(`dark_fraction`, 디스크 중앙값의 0.5배 미만), 유효 픽셀 비율(`valid_fraction`).
- **면적 180열** `suvi_t035_*`, `suvi_t045_*`, `suvi_t055_*`: 디스크 중앙값의 0.35/0.45/0.55배보다 어두운 영역(opening·closing 3×3, 면적 0.0005 이상)의 면적, 개수, 상위 3개 영역의 면적·위치·폭, 중앙 자오선 ±15°/±30°와 적도 ±30° 안의 면적, 경도×위도 36칸 면적.

계산식은 `leaderboard/model-release-publish/experiments/xgboost_suvi_fusion/{features,coronal_holes}.py`에서 옮겼고 열 이름만 바꿨습니다. 파일을 읽지 못하면 그 칸은 NaN이 되고, 파일 목록은 MANIFEST에 남깁니다.

In [4]:
SUVI_NAME = re.compile(r"_s(\d{4})(\d{3})(\d{2})(\d{2})(\d{2})(\d)_e(\d{4})(\d{3})(\d{2})(\d{2})(\d{2})(\d)_c")


def stamp(g):
    year, doy, hh, mm, ss, tenth = (int(x) for x in g)
    return pd.Timestamp(year=year, month=1, day=1, tz="UTC") + pd.Timedelta(days=doy - 1, hours=hh, minutes=mm,
                                                                             seconds=ss, milliseconds=100 * tenth)


def read_suvi_frame(path):
    # RAD를 물리 radiance로 한 번만 디코딩 (h5py, 자동 scale 없음). fill 값은 NaN
    with h5py.File(path, "r") as nc:
        var = nc["RAD"]
        raw = np.asarray(var[...])
        rad = raw.astype(np.float64)
        invalid = ~np.isfinite(rad)
        for name in ("_FillValue", "missing_value"):
            if name in var.attrs:
                invalid |= raw == np.asarray(var.attrs[name]).reshape(-1)[0]
        rad = rad * float(np.asarray(var.attrs.get("scale_factor", 1)).item())
        rad += float(np.asarray(var.attrs.get("add_offset", 0)).item())
        rad[invalid] = np.nan
        header = dict(nc.attrs)
        for key in ("CRPIX1", "CRPIX2", "RSUN", "CROTA", "SOLAR_B0", "CDELT1", "CDELT2", "PC1_1", "PC1_2", "PC2_1", "PC2_2"):
            if key in nc:
                header[key] = np.asarray(nc[key][...]).item()
        return rad.squeeze(), np.asarray(nc["DQF"][...]).squeeze(), header


def cell_stats(rad, dqf, header):
    # 디스크와 경도 12 x 위도 3 셀의 log1p radiance 통계 (148열)
    rad, dqf = np.asarray(rad, dtype=float), np.asarray(dqf)
    h = {str(k).upper(): v for k, v in header.items()}

    def number(key, default=None):
        value = h.get(key, default)
        if value is None:
            raise ValueError(f"missing SUVI geometry: {key}")
        return float(np.asarray(value).item())

    yy, xx = np.indices(rad.shape, dtype=float)
    xx -= number("CRPIX1") - 1
    yy -= number("CRPIX2") - 1
    theta = np.deg2rad(number("CROTA", 0))
    matrix = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    if all(k in h for k in ("PC1_1", "PC1_2", "PC2_1", "PC2_2")):
        matrix = np.array([[number("PC1_1"), number("PC1_2")], [number("PC2_1"), number("PC2_2")]])
    rsun = number("RSUN")
    if not np.isfinite(rsun) or rsun <= 0:
        raise ValueError("RSUN must be positive")
    x = (matrix[0, 0] * xx + matrix[0, 1] * yy) / rsun
    y = (matrix[1, 0] * xx + matrix[1, 1] * yy) / rsun
    disk = x * x + y * y <= 0.98 ** 2
    valid = disk & (dqf == 0) & np.isfinite(rad) & (rad >= 0)
    z = np.sqrt(np.maximum(0, 1 - x * x - y * y))
    b0 = np.deg2rad(number("SOLAR_B0", 0))
    lat = np.arcsin(np.clip(y * np.cos(b0) + z * np.sin(b0), -1, 1))
    lon = np.arctan2(x, z * np.cos(b0) - y * np.sin(b0))
    longitude = np.clip(np.floor((lon + np.pi / 2) / np.pi * 12), 0, 11).astype(int)
    latitude = np.clip(np.floor((lat + np.pi / 2) / np.pi * 3), 0, 2).astype(int)
    reference = np.median(rad[valid]) if valid.any() else np.nan
    result = {}
    for name, region in [("disk", disk)] + [(f"lon{i:02d}_lat{j}", disk & (longitude == i) & (latitude == j))
                                            for i in range(12) for j in range(3)]:
        values = rad[valid & region]
        log = np.log1p(values)
        result[f"suvi_{name}_log_median"] = float(np.median(log)) if len(log) else np.nan
        result[f"suvi_{name}_log_q10"] = float(np.quantile(log, .1)) if len(log) else np.nan
        result[f"suvi_{name}_dark_fraction"] = float(np.mean(values < .5 * reference)) if len(log) else np.nan
        result[f"suvi_{name}_valid_fraction"] = float(len(values) / region.sum()) if region.any() else 0.
    return result


THRESHOLDS = (0.35, 0.45, 0.55)


def dark_geometry(shape, h):
    rsun = h["RSUN"]
    if not np.isfinite(rsun) or rsun <= 0:
        raise ValueError("RSUN must be positive")
    y, x = np.indices(shape, dtype=float)
    x -= h["CRPIX1"] - 1
    y -= h["CRPIX2"] - 1
    theta = np.deg2rad(h.get("CROTA", 0))
    matrix = np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
    if all(k in h for k in ("PC1_1", "PC1_2", "PC2_1", "PC2_2")):
        matrix = np.array([[h["PC1_1"], h["PC1_2"]], [h["PC2_1"], h["PC2_2"]]])
    xx = (matrix[0, 0] * x + matrix[0, 1] * y) / rsun
    yy = (matrix[1, 0] * x + matrix[1, 1] * y) / rsun
    disk = xx ** 2 + yy ** 2 <= .98 ** 2
    z = np.sqrt(np.maximum(1e-8, 1 - xx ** 2 - yy ** 2))
    b0 = np.deg2rad(h.get("SOLAR_B0", 0))
    lat = np.rad2deg(np.arcsin(np.clip(yy * np.cos(b0) + z * np.sin(b0), -1, 1)))
    lon = np.rad2deg(np.arctan2(xx, z * np.cos(b0) - yy * np.sin(b0)))
    area = np.where(disk, 1 / (z * 2 * np.pi * rsun ** 2), 0)
    return disk, lat, lon, area


def dark_regions(rad, dqf, coordinates, threshold):
    disk, lat, lon, area = coordinates
    valid = disk & (dqf == 0) & np.isfinite(rad) & (rad >= 0)
    reference = np.median(rad[valid]) if valid.any() else np.nan
    dark = valid & (rad < threshold * reference)
    structure = np.ones((3, 3), dtype=bool)
    dark = ndimage.binary_closing(ndimage.binary_opening(dark, structure), structure) & valid
    labels, count = ndimage.label(dark, structure)
    totals = np.bincount(labels.ravel(), weights=area.ravel(), minlength=count + 1)
    keep = totals >= .0005
    keep[0] = False
    mask = keep[labels]
    labels, count = ndimage.label(mask, structure)
    return mask, labels, count, valid, lat, lon, area


def dark_area_stats(rad, dqf, header):
    # 임계 0.35/0.45/0.55 어두운 영역의 면적·위치 (180열)
    rad, dqf = np.asarray(rad, dtype=float), np.asarray(dqf)
    keys = {"CRPIX1", "CRPIX2", "RSUN", "CROTA", "SOLAR_B0", "PC1_1", "PC1_2", "PC2_1", "PC2_2"}
    h = {str(k).upper(): float(np.asarray(v).item()) for k, v in header.items() if str(k).upper() in keys}
    rsun = h["RSUN"]
    if all(key in h for key in ("PC1_1", "PC1_2", "PC2_1", "PC2_2")):
        scale = np.linalg.svd(np.array([[h["PC1_1"], h["PC1_2"]], [h["PC2_1"], h["PC2_2"]]]), compute_uv=False).min()
        if scale <= 0:
            raise ValueError("singular SUVI PC matrix")
    else:
        scale = 1.
    radius = rsun / scale + 2       # 디스크를 덮는 정사각형으로 자름 (재표본 없음, 좌표계 유지)
    cx, cy = h["CRPIX1"] - 1, h["CRPIX2"] - 1
    x0, x1 = max(0, int(np.floor(cx - radius))), min(rad.shape[1], int(np.ceil(cx + radius)) + 1)
    y0, y1 = max(0, int(np.floor(cy - radius))), min(rad.shape[0], int(np.ceil(cy + radius)) + 1)
    rad, dqf = rad[y0:y1, x0:x1], dqf[y0:y1, x0:x1]
    h["CRPIX1"], h["CRPIX2"] = cx + 1 - x0, cy + 1 - y0
    coordinates = dark_geometry(rad.shape, h)
    out = {}
    for threshold in THRESHOLDS:
        mask, labels, count, valid, lat, lon, area = dark_regions(rad, dqf, coordinates, threshold)
        result = {"area": float(area[mask].sum()), "count": float(count),
                  "valid_fraction": float(valid.sum() / max(1, (area > 0).sum()))}
        objects = []
        for label, bounds in enumerate(ndimage.find_objects(labels), 1):
            if bounds is None:
                continue
            inside = labels[bounds] == label
            weights = area[bounds][inside]
            la, lo = lat[bounds][inside], lon[bounds][inside]
            objects.append((float(weights.sum()), float(np.average(la, weights=weights)),
                            float(np.average(lo, weights=weights)), float(np.ptp(lo)), float(np.ptp(la))))
        objects.sort(key=lambda row: (-row[0], row[1], row[2]))
        result["max_area"] = objects[0][0] if objects else 0.
        for i in range(3):
            values = objects[i] if i < len(objects) else (0.,) * 5
            result.update({f"top{i + 1}_{name}": value for name, value in
                           zip(("area", "lat", "lon", "lon_width", "lat_width"), values)})
        for width in (15, 30):
            result[f"meridian{width}_area"] = float(area[mask & (np.abs(lon) <= width)].sum())
            result[f"meridian{width}_equatorial_area"] = float(area[mask & (np.abs(lon) <= width) & (np.abs(lat) <= 30)].sum())
        result["equatorial_area"] = float(area[mask & (np.abs(lat) <= 30)].sum())
        longitude = np.clip(((lon + 90) / 15).astype(int), 0, 11)
        latitude = np.clip(((lat + 90) / 60).astype(int), 0, 2)
        totals = np.bincount((longitude * 3 + latitude)[mask], weights=area[mask], minlength=36)
        result.update({f"lon{i:02d}_lat{j}_area": float(totals[i * 3 + j]) for i in range(12) for j in range(3)})
        if not valid.any():
            result = {key: np.nan for key in result}
        out.update({f"suvi_t{round(threshold * 100):03d}_{key}": value for key, value in result.items()})
    return out


def extract_one(path):
    try:
        rad, dqf, header = read_suvi_frame(path)
        res = {**cell_stats(rad, dqf, header), **dark_area_stats(rad, dqf, header)}
        return path, (tuple(res), np.fromiter(res.values(), dtype=np.float64, count=len(res))), None
    except Exception as exc:   # 읽지 못한 파일: 그 칸은 NaN, 목록은 MANIFEST에
        return path, None, f"{type(exc).__name__}: {exc}"


files = []
for p in sorted(NC_DIR.glob("*/*/*.nc")):
    m = SUVI_NAME.search(p.name)
    if m:
        files.append((str(p), stamp(m.groups()[6:])))
nc = pd.DataFrame(files, columns=["path", "obs_end"])
nc["bin"] = bin_hour(nc["obs_end"])
nc = nc.sort_values(["bin", "obs_end"])
multi = int(nc["bin"].duplicated().sum())
nc = nc.drop_duplicates("bin", keep="last")
nc = nc[nc["bin"].isin(all_origins)].reset_index(drop=True)
print(f"NetCDF {len(files):,}개 | 같은 칸의 중복 {multi} | origin 칸에 해당 {len(nc):,}개 | {nc.obs_end.min()} ~ {nc.obs_end.max()}")

t0 = time.time()
rows, errors = {}, {}
with ProcessPoolExecutor(WORKERS, mp_context=multiprocessing.get_context("fork")) as ex:
    for i, (path, res, err) in enumerate(ex.map(extract_one, nc["path"], chunksize=16), 1):
        if err:
            errors[path] = err
        else:
            rows[path] = res
        if i % 2000 == 0 or i == len(nc):
            print(f"  {i:,}/{len(nc):,} ({time.time() - t0:.0f}s)", flush=True)
cols = list(next(iter(rows.values()))[0])
assert all(k == tuple(cols) for k, _ in rows.values())
empty = np.full(len(cols), np.nan)
frame = pd.DataFrame(np.stack([rows[p][1] if p in rows else empty for p in nc["path"]]), columns=cols,
                     index=pd.DatetimeIndex(nc["bin"]))
frame.insert(0, "obs_end", pd.DatetimeIndex(nc["obs_end"]))
print(f"추출 {len(rows):,}개, 실패 {len(errors)}개, {time.time() - t0:.0f}s | 열 {len(cols)} (셀 통계 {sum(c.startswith('suvi_disk') or c.startswith('suvi_lon') for c in cols)}, 면적 {sum(c.startswith('suvi_t0') for c in cols)})")
for p, e in list(errors.items())[:10]:
    print("  실패:", p, e)

NetCDF 33,842개 | 같은 칸의 중복 0 | origin 칸에 해당 32,987개 | 2022-11-22 21:58:30.600000+00:00 ~ 2026-09-26 22:59:15.100000+00:00


  2,000/32,987 (700s)


  4,000/32,987 (1066s)


  6,000/32,987 (1460s)


  8,000/32,987 (1839s)


  10,000/32,987 (2202s)


... (9 outputs left out)

  30,000/32,987 (6054s)


  32,000/32,987 (6393s)


  32,987/32,987 (6542s)


추출 32,983개, 실패 4개, 6548s | 열 328 (셀 통계 148, 면적 180)
  실패: /home/t-lab01/sunrun/data-v2/raw/SUVI/nc/g18/2025/340/OR_SUVI-L1b-Fe195_G18_s20253400856275_e20253400856285_c20253400859414.nc ValueError: RSUN must be positive
  실패: /home/t-lab01/sunrun/data-v2/raw/SUVI/nc/g18/2025/340/OR_SUVI-L1b-Fe195_G18_s20253401056279_e20253401056289_c20253401059409.nc ValueError: RSUN must be positive
  실패: /home/t-lab01/sunrun/data-v2/raw/SUVI/nc/g18/2025/340/OR_SUVI-L1b-Fe195_G18_s20253401256282_e20253401256292_c20253401259429.nc ValueError: RSUN must be positive
  실패: /home/t-lab01/sunrun/data-v2/raw/SUVI/nc/g18/2025/340/OR_SUVI-L1b-Fe195_G18_s20253401456286_e20253401456296_c20253401459433.nc ValueError: RSUN must be positive


## 4. SUVI 피처

- origin O는 **칸 O의 프레임만** 씁니다(`obs_end` ∈ [O−1h, O)). 칸 O에 프레임이 없으면 SUVI 열이 모두 NaN이고, 이전 프레임으로 되돌아가지 않습니다.
- `img_age_h = O − obs_end`(시간). 칸 규칙 때문에 0 초과 1 이하입니다.
- **strip 15열** `strip_v{350,450,550,650,750}_{logmed,dark,cover}`는 h마다 다릅니다. 속도 v의 태양풍이 target 시각 O+h에 L1에 닿으려면 떠난 경도가 λ(h, v) = 0.55°/h × (τ_v − (h + img_age_h))이고, τ_v = (1 AU − 2.5 R☉)/v입니다. 그 경도를 중심으로 폭 10° strip이 위도 가운데 칸(`lat1`) 경도 12칸과 겹치는 비율로 `log_median`·`dark_fraction`·`valid_fraction`을 가중 평균합니다. 계산식은 `d1_longitude_strips/strips.py`에서 옮겼고, cover < 0.5 마스크와 6h stale 마스크는 뺐습니다. strip이 디스크 밖(겹침 0)이면 NaN입니다.

In [5]:
OMEGA, D_KM, CELL_W, STRIP_HALF = 0.55, 147858620.7, 15.0, 5.0
SPEEDS = (350, 450, 550, 650, 750)
TAU = np.array([D_KM / v / 3600.0 for v in SPEEDS])
STRIP_STATS = {"logmed": "log_median", "dark": "dark_fraction", "cover": "valid_fraction"}
STRIP_COLS = [f"strip_v{v}_{s}" for v in SPEEDS for s in STRIP_STATS]
H_ALL = np.arange(1, H + 1)
EDGES = -90.0 + CELL_W * np.arange(13)

suvi_feat = frame.reindex(all_origins)
img_age_h = ((all_origins - pd.DatetimeIndex(suvi_feat["obs_end"])) / HOUR).to_numpy(dtype=float)
suvi_feat = suvi_feat.drop(columns="obs_end")
suvi_feat.insert(0, "img_age_h", img_age_h)
LAT1 = {s: suvi_feat[[f"suvi_lon{i:02d}_lat1_{suffix}" for i in range(12)]].to_numpy(float) for s, suffix in STRIP_STATS.items()}


def strip_block(idx):
    # origin 위치 idx (n,) -> (n*72, 15) strip 값
    age = img_age_h[idx]
    lam = OMEGA * (TAU[None, None, :] - (H_ALL[None, :, None] + age[:, None, None]))           # (n, 72, 5)
    w = np.clip(np.minimum(lam[..., None] + STRIP_HALF, EDGES[1:]) - np.maximum(lam[..., None] - STRIP_HALF, EDGES[:-1]),
                0, None) / (2 * STRIP_HALF)                                                    # (n, 72, 5, 12)
    ws = w.sum(-1)
    out = np.empty((len(idx), H, len(SPEEDS), 3))
    for k, s in enumerate(STRIP_STATS):
        num = (w * LAT1[s][idx][:, None, None, :]).sum(-1)
        out[..., k] = np.where(ws > 0, num / np.where(ws > 0, ws, 1), np.nan)
    return out.reshape(len(idx) * H, len(STRIP_COLS))


have = np.isfinite(img_age_h)
print(f"SUVI 프레임이 있는 origin: {have.sum():,}/{len(all_origins):,} ({have.mean():.1%}) | img_age_h 범위 {np.nanmin(img_age_h):.3f}~{np.nanmax(img_age_h):.3f}h")

SUVI 프레임이 있는 origin: 32,987/34,968 (94.3%) | img_age_h 범위 0.006~0.998h


## 5. ENLIL 피처

`raw/ENLIL/donki/`의 WSA-ENLIL 실행 중 `estimatedShockArrivalTime`이 있는 것만 씁니다. 실행 하나 = (`modelCompletionTime`, 도착 예상 시각, max(kp_90, kp_135, kp_180), `isEarthGB`)이고, 같은 내용의 실행은 하나로 칩니다.
- 실행은 칸 `bin_hour(modelCompletionTime)`에 들어가고, origin O는 칸이 [O−7일, O]인 실행을 씁니다.
- dt = 도착 예상 − O(시간). 앞으로 올 것 = dt ≥ −6h, 지난 것 = −96h ≤ dt < −6h.
- 이 기준으로 다음 열을 만듭니다.
  - `enlil_next_arr_h`: 앞으로 올 것의 최소 dt
  - `enlil_since_arr_h`: 지난 것의 −최대 dt
  - `enlil_n_pending`: 앞으로 올 것의 개수
  - `enlil_kp`: 앞으로 올 것의 최대 kp, 없으면 0
  - `enlil_glancing`: 앞으로 올 것이 모두 glancing blow인지
- h마다 다른 열은 `enlil_dt_target = h − enlil_next_arr_h`입니다.

In [6]:
runs = []
for p in sorted(ENLIL_DIR.glob("WSAEnlilSimulations_*.json")):
    for s in json.loads(p.read_text()):
        if s.get("estimatedShockArrivalTime"):
            runs.append((pd.Timestamp(s["modelCompletionTime"]), pd.Timestamp(s["estimatedShockArrivalTime"]),
                         max(s.get("kp_90") or 0, s.get("kp_135") or 0, s.get("kp_180") or 0), bool(s.get("isEarthGB"))))
runs = pd.DataFrame(runs, columns=["done", "arrival", "kp", "glancing"]).drop_duplicates()
runs["bin"] = bin_hour(runs["done"])
runs = runs.sort_values("bin").reset_index(drop=True)

rb = runs["bin"].to_numpy("datetime64[ns]").astype(np.int64)
ra = pd.DatetimeIndex(runs["arrival"]).tz_convert("UTC").to_numpy("datetime64[ns]").astype(np.int64)
rk, rg = runs["kp"].to_numpy(float), runs["glancing"].to_numpy(bool)
on = all_origins.to_numpy("datetime64[ns]").astype(np.int64)
NS_H, WEEK = 3600 * 10 ** 9, 7 * 24 * 3600 * 10 ** 9
enl = np.full((len(on), 5), np.nan)
hi = np.searchsorted(rb, on, side="right")
lo = np.searchsorted(rb, on - WEEK, side="left")
for i in range(len(on)):
    dt = (ra[lo[i]:hi[i]] - on[i]) / NS_H
    fut, past = dt >= -6, (dt < -6) & (dt >= -96)
    enl[i] = [dt[fut].min() if fut.any() else np.nan, -dt[past].max() if past.any() else np.nan, fut.sum(),
              rk[lo[i]:hi[i]][fut].max() if fut.any() else 0, float(rg[lo[i]:hi[i]][fut].all()) if fut.any() else np.nan]
ENLIL_COLS = ["enlil_next_arr_h", "enlil_since_arr_h", "enlil_n_pending", "enlil_kp", "enlil_glancing"]
enlil_feat = pd.DataFrame(enl, index=all_origins, columns=ENLIL_COLS)
print(f"ENLIL 실행 {len(runs):,}개 ({runs.done.min()} ~ {runs.done.max()}) | 앞으로 올 실행이 있는 origin {(enlil_feat.enlil_n_pending > 0).mean():.1%}")

ENLIL 실행 964개 (2022-07-04 20:17:00+00:00 ~ 2026-10-21 15:31:00+00:00) | 앞으로 올 실행이 있는 origin 57.5%


## 6. df 조립·저장

- 형식은 예전 df와 같은 (`set`, `origin`, `h`) 긴 형식이고, origin 하나가 h = 1…72의 72행입니다.
  - `set = train`: origin 2022-10-01T00Z ~ 2026-06-23T00Z(F3 cutoff − 144h), 1h 간격.
  - `set = query`: origin 2025-12-31T23Z ~ 2026-09-26T23Z, 1h 간격, 6,457개. `period`는 origin이 속한 F1/F2/F3이고, 기간 사이 71개 origin은 비어 있습니다.
  - 같은 origin이 두 set에 모두 있으면 값은 같고 `set`만 다릅니다.
- `target_time = origin + h`. `y`는 1단계의 T 라벨 status 0 속도입니다.
- `fit_P = (set == train) & (origin + 144h ≤ cutoff_P)`, cutoff는 F1 2025-12-29, F2 2026-03-29, F3 2026-06-29입니다.
- 열 순서는 메타 9열, 상수 피처(ACE, SUVI, ENLIL), h마다 다른 피처(`rec26/27/28`, `rec_mean`, strip 15열, `enlil_dt_target`)입니다.
  - `rec{d}`는 ACE 속도 칸(target_time − d일)이고, d = 26/27/28은 27일 태양 자전 재귀입니다. h ≤ 72이므로 그 칸은 항상 origin 이전입니다.
  - `rec_mean`은 셋 중 NaN이 아닌 값의 평균입니다.
- 메모리를 아끼려고 origin 1,000개씩 펼쳐 row group으로 이어 씁니다(zstd). 숫자 피처는 float32, `y`는 float64입니다.

In [7]:
const = pd.concat([ace_feat, suvi_feat, enlil_feat], axis=1)
assert const.index.equals(all_origins) and const.columns.is_unique
CONST_COLS = list(const.columns)
VARY_COLS = ["rec26", "rec27", "rec28", "rec_mean"] + STRIP_COLS + ["enlil_dt_target"]
CONST = const.to_numpy(np.float32)
pos = pd.Series(np.arange(len(all_origins)), index=all_origins)
speed_bin = ace_bin["ace_speed_kms"]
next_arr = enlil_feat["enlil_next_arr_h"].to_numpy(float)

SET_DICT = pa.array(["train", "query"])
PERIOD_DICT = pa.array(list(PERIODS))
period_of = pd.Series(pd.NA, index=query_origins, dtype="object")
for name, (first, last, _) in PERIODS.items():
    period_of[(query_origins >= pd.Timestamp(first)) & (query_origins <= pd.Timestamp(last))] = name
CUTOFF = {p: pd.Timestamp(c) for p, (_, _, c) in PERIODS.items()}


def dict_col(values, dictionary):
    lookup = {v: i for i, v in enumerate(dictionary.to_pylist())}
    idx = pa.array([lookup.get(v) if isinstance(v, str) else None for v in values], type=pa.int8())
    return pa.DictionaryArray.from_arrays(idx, dictionary)


def block(set_name, origins):
    n = len(origins)
    idx = pos[origins].to_numpy()
    o = np.repeat(origins.to_numpy("datetime64[ns]"), H)
    h = np.tile(H_ALL, n).astype(np.int16)
    tt = o + h.astype("timedelta64[h]").astype("timedelta64[ns]")
    ttd = pd.DatetimeIndex(tt, tz="UTC")
    od = pd.DatetimeIndex(o, tz="UTC")
    periods = np.repeat(period_of.reindex(origins).to_numpy() if set_name == "query" else np.full(n, None), H)
    cols = {"set": dict_col([set_name] * (n * H), SET_DICT),
            "origin": pa.array(od, type=pa.timestamp("ns", tz="UTC")),
            "h": pa.array(h),
            "target_time": pa.array(ttd, type=pa.timestamp("ns", tz="UTC")),
            "period": dict_col(periods, PERIOD_DICT)}
    for p, cut in CUTOFF.items():
        cols[f"fit_{p}"] = pa.array((set_name == "train") & (od + pd.Timedelta(hours=MASK_HOURS) <= cut))
    cols["y"] = pa.array(y_series.reindex(ttd).to_numpy(float))
    c = np.repeat(CONST[idx], H, axis=0)
    for j, name in enumerate(CONST_COLS):
        cols[name] = pa.array(c[:, j])
    rec = np.stack([speed_bin.reindex(ttd - pd.Timedelta(days=d)).to_numpy(float) for d in (26, 27, 28)], 1)
    vary = [rec[:, 0], rec[:, 1], rec[:, 2], np.nanmean(rec, 1)]
    st = strip_block(idx)
    vary += [st[:, k] for k in range(st.shape[1])]
    vary.append(h - np.repeat(next_arr[idx], H))
    for name, val in zip(VARY_COLS, vary):
        cols[name] = pa.array(np.asarray(val, dtype=np.float32))
    return pa.table(cols)


OUT.mkdir(parents=True, exist_ok=True)
path, tmp = OUT / "df.parquet", OUT / "df.parquet.tmp"
t0, writer, n_rows = time.time(), None, {}
try:
    for set_name, origins in (("train", train_origins), ("query", query_origins)):
        n_rows[set_name] = 0
        for s in range(0, len(origins), 1000):
            tbl = block(set_name, origins[s:s + 1000])
            if writer is None:
                writer = pq.ParquetWriter(tmp, tbl.schema, compression="zstd")
            writer.write_table(tbl)
            n_rows[set_name] += tbl.num_rows
        print(f"  {set_name}: {n_rows[set_name]:,}행 ({time.time() - t0:.0f}s)", flush=True)
finally:
    if writer is not None:
        writer.close()
tmp.replace(path)
columns = tbl.schema.names
print(f"{path}: {sum(n_rows.values()):,}행 × {len(columns)}열, {path.stat().st_size / 1e9:.2f} GB, {time.time() - t0:.0f}s")

  train: 2,351,880행 (30s)


  query: 464,904행 (36s)


/home/t-lab01/sunrun/data-v2/input/df.parquet: 2,816,784행 × 446열, 0.21 GB, 36s


In [8]:
q_per = period_of.dropna().value_counts().sort_index()
manifest = {
    "dataset": "input-df",
    "description": "Flat (set, origin, h) model-input table built only from data-v2/raw: every raw-derived feature, "
                   "hourly train and query origins, y = ACE SWEPAM speed with status 0 (no fill), F1/F2/F3 144 h masks.",
    "built_by": "data-v2/notebooks/raw_to_input_df.ipynb",
    "built_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "smoke": SMOKE,
    "rules": {
        "time_bin": "a time t goes to hour bin floor(t, 1h) + 1h ([H-1h, H) -> H); latest record per bin; origin O uses bins <= O",
        "y": "SWEPAM bulk speed at the label time T = target_time (no shift), status == 0 and finite > 0, else NaN, no fill",
        "ace_features": "value is NaN when its status != 0 or it equals the product missing value; no fill, no log, no clip",
        "suvi": "only the frame in bin O (obs_end in [O-1h, O)); no walk-back, no stale or cover mask",
        "enlil": "runs with bin(modelCompletionTime) in [O-7d, O]",
        "processing": "no availability delay, missing-value or outlier handling",
    },
    "rows": int(sum(n_rows.values())),
    "rows_by_set": n_rows,
    "columns": columns,
    "constant_features": len(CONST_COLS),
    "varying_features": len(VARY_COLS),
    "train_origins": len(train_origins), "train_span": [str(train_origins[0]), str(train_origins[-1])],
    "query_origins": len(query_origins), "query_span": [str(query_origins[0]), str(query_origins[-1])],
    "query_origins_by_period": {k: int(v) for k, v in q_per.items()},
    "sources": {
        "ace": {"dir": str(ACE_DIR.relative_to(DATA)), **{k: v for k, v in ace_info.items()},
                "labels": [str(ace_label.index.min()), str(ace_label.index.max())]},
        "suvi": {"dir": str(NC_DIR.relative_to(DATA)), "files_used": int(len(nc)), "files_same_bin_dropped": multi,
                 "read_errors": errors, "obs_end": [str(nc.obs_end.min()), str(nc.obs_end.max())]},
        "enlil": {"dir": str(ENLIL_DIR.relative_to(DATA)), "runs": int(len(runs))},
    },
    "files": {"df.parquet": {"bytes": path.stat().st_size}},
}
(OUT / "MANIFEST.json").write_text(json.dumps(manifest, indent=1, default=str) + "\n")
print("MANIFEST.json 저장 |", manifest["rows_by_set"], "| 상수", len(CONST_COLS), "| h별", len(VARY_COLS))

MANIFEST.json 저장 | {'train': 2351880, 'query': 464904} | 상수 417 | h별 20


## 7. 검증

In [9]:
pf = pq.ParquetFile(path)
assert pf.schema_arrow.names == columns and pf.metadata.num_rows == sum(n_rows.values())
meta = pd.read_parquet(path, columns=["set", "origin", "h", "target_time", "period", "fit_F1", "fit_F2", "fit_F3", "y"])
q, tr = meta[meta["set"] == "query"], meta[meta["set"] == "train"]

# (a) origin 수와 간격: set마다 1h 연속, origin당 h = 1..72
for name, part, want in (("train", tr, train_origins), ("query", q, query_origins)):
    o = part["origin"].drop_duplicates()
    assert len(o) == len(want) and (o.diff().dropna() == HOUR).all(), name
    assert (part.groupby("origin")["h"].agg(lambda s: (s.to_numpy() == H_ALL).all())).all(), name
if not SMOKE:
    per = q.groupby("period", observed=True)["origin"].nunique().to_dict()
    assert per == {"F1": 2089, "F2": 2113, "F3": 2113}, per
print("(a) origin: train", tr["origin"].nunique(), "| query", q["origin"].nunique(), q.groupby("period", observed=True)["origin"].nunique().to_dict())

# (b) target_time = origin + h, y = target_time 라벨의 SWEPAM status 0 속도 (이동·채움 없음)
assert (meta["target_time"] - meta["origin"] == pd.to_timedelta(meta["h"], unit="h")).all()
sw = ace_raw["swepam"]
ok0 = sw["S"].eq(0) & (sw["ace_speed_kms"] > 0)
want_y = sw["ace_speed_kms"].where(ok0).reindex(pd.DatetimeIndex(meta["target_time"])).to_numpy()
assert np.array_equal(meta["y"].to_numpy(), want_y, equal_nan=True)
print(f"(b) y: target_time 라벨 status 0 속도와 일치 | NaN {meta['y'].isna().mean():.2%}")

# (c) fit 마스크: train 행만, origin + 144h <= cutoff, 학습 타깃은 그 기간 첫 query origin 이전
for p, cut in CUTOFF.items():
    used = meta[meta[f"fit_{p}"]]
    assert used["set"].eq("train").all() and (used["origin"] + pd.Timedelta(hours=MASK_HOURS) <= cut).all()
    first_q = pd.Timestamp(PERIODS[p][0])
    assert (used["target_time"] < first_q).all()
    print(f"(c) {p}: 학습 origin {used['origin'].nunique():,}개 (마지막 {used['origin'].max()}), cutoff {cut}")

# (d) 인과성: 표본 origin에서 원천 시각이 origin보다 앞서는지 raw에서 다시 확인
rng = np.random.default_rng(0)
sample = pd.DatetimeIndex(rng.choice(all_origins, size=min(300, len(all_origins)), replace=False))
cols_d = ["origin", "h", "ace_speed_kms_last", "ace_bt_nt_m6", "img_age_h", "enlil_n_pending", "rec27", "strip_v450_logmed"]
part = (pd.read_parquet(path, columns=cols_d, filters=[("h", "=", 1)]).drop_duplicates("origin")
        .set_index("origin").loc[sample].sort_index())
lab = ace_label.reindex(part.index - HOUR)                          # 칸 O = 라벨 O-1h
assert np.allclose(part["ace_speed_kms_last"], lab["ace_speed_kms"], equal_nan=True)
win = [ace_label["ace_bt_nt"].reindex(pd.date_range(o - 6 * HOUR, o - HOUR, freq="h")).mean() for o in part.index]
assert np.allclose(part["ace_bt_nt_m6"], np.asarray(win, dtype=np.float32), equal_nan=True)
age = part["img_age_h"].dropna()
assert ((age > 0) & (age <= 1)).all()                               # obs_end in [O-1h, O)
n_pend = [int(((runs["bin"] <= o) & (runs["bin"] >= o - pd.Timedelta(days=7)) &
               ((runs["arrival"] - o) / HOUR >= -6)).sum()) for o in part.index]
assert np.array_equal(part["enlil_n_pending"].to_numpy(), np.asarray(n_pend, dtype=np.float32))
rec_src = ace_label["ace_speed_kms"].reindex(part.index + HOUR - pd.Timedelta(days=27) - HOUR)
assert np.allclose(part["rec27"], rec_src, equal_nan=True) and (part.index + HOUR - pd.Timedelta(days=27) <= part.index).all()
print(f"(d) 표본 {len(part)} origin: ACE last·m6, img_age_h ∈ (0, 1], enlil_n_pending, rec27이 raw 재계산과 일치")

# (e) 이름: ch 토큰 없음, 열 중복 없음
assert not [c for c in columns if re.search(r"(^|_)ch(_|$)", c.lower())]
assert len(set(columns)) == len(columns)
print("(e) 열", len(columns), "| ch 이름 없음")

# (f) 결측 비율 (처리하지 않은 그대로)
nan_rate = pd.read_parquet(path, columns=CONST_COLS + VARY_COLS, filters=[("h", "=", 1)]).isna().mean()
display(nan_rate.groupby(lambda c: "ace" if c.startswith(("ace_", "v_", "speed_", "epam_")) else
                         "suvi" if c.startswith(("suvi_", "img_", "strip_")) else "rec" if c.startswith("rec") else "enlil")
        .describe().round(3))

(a) origin: train 32665 | query 6457 {'F1': 2089, 'F2': 2113, 'F3': 2113}


(b) y: target_time 라벨 status 0 속도와 일치 | NaN 32.18%
(c) F1: 학습 origin 28,297개 (마지막 2025-12-23 00:00:00+00:00), cutoff 2025-12-29 00:00:00+00:00


(c) F2: 학습 origin 30,457개 (마지막 2026-03-23 00:00:00+00:00), cutoff 2026-03-29 00:00:00+00:00
(c) F3: 학습 origin 32,665개 (마지막 2026-06-23 00:00:00+00:00), cutoff 2026-06-29 00:00:00+00:00


(d) 표본 300 origin: ACE last·m6, img_age_h ∈ (0, 1], enlil_n_pending, rec27이 raw 재계산과 일치
(e) 열 446 | ch 이름 없음


,count,mean,std,min,25%,50%,75%,max
ace,83.0,0.114,0.208,0.000,0.000,0.009,0.168,1.000
enlil,6.0,0.272,0.213,0.000,0.088,0.388,0.427,0.427
rec,4.0,0.279,0.087,0.148,0.279,0.323,0.323,0.323
suvi,344.0,0.066,0.067,0.053,0.053,0.053,0.053,0.466


## 8. 사용법

```python
import pandas as pd
DF = "/home/t-lab01/sunrun/data-v2/input/df.parquet"
cols = ["set", "origin", "h", "period", "fit_F1", "y", "v_lag0", "suvi_t035_equatorial_area", "strip_v450_logmed"]
df = pd.read_parquet(DF, columns=cols)                      # 전체 약 2.8M행 x 450열이므로 필요한 열만 읽기
fit_f1 = df[df.fit_F1]                                       # F1 학습 행
query_f1 = df[(df.set == "query") & (df.period == "F1")]    # F1 예측 origin 2,089개 x 72h
```

## 9. 한계와 주의

- 값이 예전 df(2026-11-01)와 다릅니다.
  - 시간 칸 규칙 때문에 ACE 칸 O는 라벨 O−1h 행입니다.
  - SUVI·ENLIL에 되돌아가기와 마스크가 없습니다.
  - `y`는 status 0만 쓰고 채우지 않습니다.
  - train origin이 3h가 아니라 1h 간격입니다.
  - EPAM/SIS에 log10을 쓰지 않습니다.
  - 예전 `ch_ch_*` 열은 `suvi_*`로 바뀌었고 `img_stale`은 없습니다.
- `research/e2000fs_calib_point4_144h/experiment.py`는 예전 열 이름(`ch_ch_*`)을 읽으므로 그대로는 실행되지 않습니다.
- 원문은 받은 시점의 최신 값입니다. 각 시점에 실제로 공개돼 있던 값인지는 검증되지 않았습니다.
- 2022-08-22~11-21에는 SUVI 파일이 없습니다. 그 칸의 SUVI 피처는 NaN입니다.
- 생성 파일 `data-v2/input/{df.parquet, MANIFEST.json}`은 데이터라서 git에 올리지 않습니다.